# INEBase

INEBase utiliza el estándar [OpenAPI](https://www.openapis.org/) (anteriormente Swagger). Lo llaman [WSTempus](https://www.ine.es/OpenAPI/index.html#/).

"Toda la información publicada por el INE, independientemente de la fuente, va a poder ser consultada a través de estas diferentes APIs."

Los datos se pueden conseguir en español (ES) o inglés (EN).

La explicación más clara es la de "[Referencia de la API](https://www.ine.es/dyngs/DAB/index.htm?cid=1100)".

Estructura de la URL: `https://servicios.ine.es/wstempus/js/{idioma}/{función}/{input}[?parámetros]`

In [ ]:
import pandas as pd
import requests

from pathlib import Path

PATH_DATA = Path("../data")

Hay 3 tipos de tablas: Tempus3, PC-Axis y tpx

Tempus3 se puede considerar que es la más sofisticada. A continuación se investiga algo más a fondo.

El nivel jerárquico más alto del INEBase son las operaciones:

In [2]:
url = "https://servicios.ine.es/wstempus/js/ES/OPERACIONES_DISPONIBLES"

response = requests.get(url)
response.raise_for_status()

In [17]:
# Investigamos qué devuelve el servidor
for i, line in enumerate(response.text.splitlines()[:5]):
    print(i, repr(line))
print("Filas:", len(response.text.splitlines()))

0 '[{"Id":6, "Cod_IOE":"30211", "Nombre":"Índice de Coste Laboral Armonizado", "Descripcion":"El Indice de Coste Laboral Armonizado es una operación estadística continua de periodicidad trimestral cuyo objetivo fundamental es proporcionar una medida común, comparable y oportuna de los costes laborales para toda la Unión Europea, que permita un seguimiento de la evolución de dichos costes laborales.\\nPara la obtención del índice de costes laborales armonizado se utiliza el índice Laspeyres, encadenado anualmente y basado en una estructura fija de la actividad económica, desglosada por secciones de la CNAE-09. Se calculan índices para las siguientes variables por hora efectiva de trabajo: el coste laboral total, coste salarial total, otros costes y el coste laboral excluyendo pagos extraordinarios y atrasos.", "Codigo":"ICLA", "Url":"/dyngs/INEbase/operacion.htm?c=Estadistica_C&cid=1254736053992&idp=1254735976596"}'
1 ',{"Id":7, "Cod_IOE":"30168", "Nombre":"Estadística de Transmisión de

A fecha de hacer el análisis, hay más de 160 operaciones.

Podemos intentar filtrarlas:

In [12]:
# Usar response.json devuelve una lista de diccionarios. Cada diccionario es una operación.
response.json()[:2]

[{'Id': 6,
  'Cod_IOE': '30211',
  'Nombre': 'Índice de Coste Laboral Armonizado',
  'Descripcion': 'El Indice de Coste Laboral Armonizado es una operación estadística continua de periodicidad trimestral cuyo objetivo fundamental es proporcionar una medida común, comparable y oportuna de los costes laborales para toda la Unión Europea, que permita un seguimiento de la evolución de dichos costes laborales.\nPara la obtención del índice de costes laborales armonizado se utiliza el índice Laspeyres, encadenado anualmente y basado en una estructura fija de la actividad económica, desglosada por secciones de la CNAE-09. Se calculan índices para las siguientes variables por hora efectiva de trabajo: el coste laboral total, coste salarial total, otros costes y el coste laboral excluyendo pagos extraordinarios y atrasos.',
  'Codigo': 'ICLA',
  'Url': '/dyngs/INEbase/operacion.htm?c=Estadistica_C&cid=1254736053992&idp=1254735976596'},
 {'Id': 7,
  'Cod_IOE': '30168',
  'Nombre': 'Estadística d

In [15]:
# Buscamos operaciones que contienen "oblaci" en el nombre, esperando encontrar "población", "Población" y sin que importe la tilde.
filtro = "oblaci"
operaciones = [operacion for operacion in response.json() if filtro in operacion["Nombre"]]
operaciones

[{'Id': 8,
  'Cod_IOE': '30243',
  'Nombre': 'Censos de Población y Viviendas',
  'Descripcion': 'Censo de Población y Viviendas 2021\nEl Censo de Población es una operación estadística que tiene como objetivos:\nRecuento de la población y los hogares .\nEl Censo de Población determina el número de habitantes de España y de todas sus divisiones administrativas (las comunidades autónomas, provincias, municipios e incluso distritos y secciones censales) a 1 de enero de 2021. Esta operación va dirigida a las personas que residen en viviendas (ya sean viviendas familiares convencionales o alojamientos) o en establecimientos colectivos (hoteles, residencias, asilos...).\nConocimiento de la estructura de la población y los hogares .\nEl Censo de Población no solo cuenta el número de habitantes sino que investiga, para cada uno de ellos, una serie de características demográficas, culturales, económicas y sociales y proporciona una imagen de la estructura de la población.\nEl Censo de Vivienda

In [14]:
# Cogemos una operación
filtro = "Estadística Continua de Población"
operacion = [operacion for operacion in response.json() if filtro in operacion["Nombre"]] 
operacion

[{'Id': 450,
  'Cod_IOE': '30282',
  'Nombre': 'Estadística Continua de Población',
  'Descripcion': 'La Estadística Continua de Población contiene:\n- La población residente en España, total y en viviendas familiares, en cada comunidad autónoma, en cada provincia y en cada isla, desagregadas según características demográficas básicas. Además, a partir de 2026 en la publicación de mayo se publican datos a 1 de enero para los principales municipios y capitales de provincia.\n- El número de hogares en que reside la población en viviendas familiares según tamaño (número de miembros de hogar), en cada provincia.\nEstos datos son empleados como cifras poblacionales de referencia en todas las operaciones estadísticas del INE (encuestas, Contabilidad Nacional, indicadores, etc.) y son las transmitidas a nivel internacional como datos oficiales de población de España a todos los efectos.\nPara la población residente se ofrecen resultados desde 2021, enlazando con los resultados de los sucesivo

Podemos buscar las tablas relacionadas con una operación:

In [ ]:
codigo_operacion = "ECP"

url = f"https://servicios.ine.es/wstempus/js/ES/TABLAS_OPERACION/{codigo_operacion}"

response = requests.get(url)
response.raise_for_status()

In [21]:
# Investigamos qué devuelve el servidor
for i, line in enumerate(response.text.splitlines()[:5]):
    print(i, repr(line))

0 '[{"Id":56934, "Nombre":"Población residente por fecha, sexo y edad (desde 1971)", "Codigo":"NAC_SDET", "FK_Periodicidad":3, "FK_Publicacion":610, "FK_Periodo_ini":19, "Anyo_Periodo_ini":"1971", "FK_Periodo_fin":19, "Anyo_Periodo_fin":"2025", "Ultima_Modificacion":"Tue Dec 02 11:10:00 CET 2025"}'
1 ',{"Id":56940, "Nombre":"Población residente por fecha, sexo y edad (desde 1971)", "Codigo":"CCAA_SDET", "FK_Periodicidad":3, "FK_Publicacion":610, "FK_Periodo_ini":19, "Anyo_Periodo_ini":"1971", "FK_Periodo_fin":19, "Anyo_Periodo_fin":"2025", "Ultima_Modificacion":"Tue Dec 02 11:10:00 CET 2025"}'
2 ',{"Id":56945, "Nombre":"Población residente por fecha, sexo y edad (desde 1971)", "Codigo":"PROV_SDET", "FK_Periodicidad":3, "FK_Publicacion":610, "FK_Periodo_ini":19, "Anyo_Periodo_ini":"1971", "FK_Periodo_fin":19, "Anyo_Periodo_fin":"2025", "Ultima_Modificacion":"Tue Dec 02 11:10:00 CET 2025"}'
3 ',{"Id":59011, "Nombre":"Inmigración procedente del extranjero, por trimestre y nacionalidad (10

Podemos seleccionar una tabla e investigarla.

En este caso es necesario limitar la consulta.

### Filtro de una tabla de Tempus3

1. Conocer los grupos de la tabla de interés

In [45]:
codigo_tabla = "56945"

url = f"https://servicios.ine.es/wstempus/js/ES/GRUPOS_TABLA/{codigo_tabla}"

response = requests.get(url)
response.raise_for_status()
response.json()

[{'Id': 113388, 'Nombre': 'Edad simple'},
 {'Id': 113389, 'Nombre': 'Provincias'},
 {'Id': 113390, 'Nombre': 'Sexo'}]

2. Conocer los valores de los grupos

In [47]:
codigos_valores = [valor["Id"] for valor in response.json()]
for codigo_valor in codigos_valores:
    url = f"https://servicios.ine.es/wstempus/js/ES/VALORES_GRUPOSTABLA/{codigo_tabla}/{codigo_valor}?det=1"
    display(requests.get(url).json())


[{'Id': 15668,
  'Variable': {'Id': 356, 'Nombre': 'Totales de edad', 'Codigo': ''},
  'Nombre': 'Todas las edades',
  'Codigo': ''},
 {'Id': 15319,
  'Variable': {'Id': 355, 'Nombre': 'Valores simples de edad', 'Codigo': ''},
  'Nombre': '0 años',
  'Codigo': 'Y0',
  'JerarquiaPadres': [{'Id': 10563,
    'FK_Variable': 360,
    'Nombre': 'De 0 a 15 años',
    'Codigo': 'Y0T15'},
   {'Id': 15073,
    'FK_Variable': 360,
    'Nombre': 'De 0 a 4 años',
    'Codigo': 'Y0T4'}]},
 {'Id': 15320,
  'Variable': {'Id': 355, 'Nombre': 'Valores simples de edad', 'Codigo': ''},
  'Nombre': '1 año',
  'Codigo': 'Y1',
  'JerarquiaPadres': [{'Id': 10563,
    'FK_Variable': 360,
    'Nombre': 'De 0 a 15 años',
    'Codigo': 'Y0T15'},
   {'Id': 15073,
    'FK_Variable': 360,
    'Nombre': 'De 0 a 4 años',
    'Codigo': 'Y0T4'},
   {'Id': 15659,
    'FK_Variable': 360,
    'Nombre': 'De 1 a 4 años   ',
    'Codigo': 'Y1T4'}]},
 {'Id': 15321,
  'Variable': {'Id': 355, 'Nombre': 'Valores simples de edad',

[{'Id': 16473,
  'Variable': {'Id': 349, 'Nombre': 'Total Nacional', 'Codigo': 'NAC'},
  'Nombre': 'Total Nacional',
  'Codigo': '00'},
 {'Id': 3,
  'Variable': {'Id': 115, 'Nombre': 'Provincias', 'Codigo': 'PROV'},
  'Nombre': 'Albacete',
  'Codigo': '02',
  'JerarquiaPadres': [{'Id': 9004,
    'FK_Variable': 70,
    'Nombre': 'Castilla - La Mancha',
    'Codigo': '08'}]},
 {'Id': 4,
  'Variable': {'Id': 115, 'Nombre': 'Provincias', 'Codigo': 'PROV'},
  'Nombre': 'Alicante/Alacant',
  'Codigo': '03',
  'JerarquiaPadres': [{'Id': 9006,
    'FK_Variable': 70,
    'Nombre': 'Comunitat Valenciana',
    'Codigo': '10'}]},
 {'Id': 5,
  'Variable': {'Id': 115, 'Nombre': 'Provincias', 'Codigo': 'PROV'},
  'Nombre': 'Almería',
  'Codigo': '04',
  'JerarquiaPadres': [{'Id': 8997,
    'FK_Variable': 70,
    'Nombre': 'Andalucía',
    'Codigo': '01'}]},
 {'Id': 2,
  'Variable': {'Id': 115, 'Nombre': 'Provincias', 'Codigo': 'PROV'},
  'Nombre': 'Araba/Álava',
  'Codigo': '01',
  'JerarquiaPadres':

[{'Id': 451,
  'Variable': {'Id': 18, 'Nombre': 'Sexo', 'Codigo': ''},
  'Nombre': 'Total',
  'Codigo': ''},
 {'Id': 452,
  'Variable': {'Id': 18, 'Nombre': 'Sexo', 'Codigo': ''},
  'Nombre': 'Hombres',
  'Codigo': '1'},
 {'Id': 453,
  'Variable': {'Id': 18, 'Nombre': 'Sexo', 'Codigo': ''},
  'Nombre': 'Mujeres',
  'Codigo': '6'}]

3. Filtrar por la variables y valores que nos interesan 

In [64]:
codigo_tabla = "56945"

url = f"https://servicios.ine.es/wstempus/js/ES/DATOS_TABLA/{codigo_tabla}"

params = [
    ("nult", 1),
    ("tip", "A"),

    ("tv", "356:15668"),   # Todas las edades
    ("tv", "349:16473"),   # Total nacional
    ("tv", "18:451"),    # Total (Hombres + Mujeres)
]

response = requests.get(url, params=params)
response.raise_for_status()
response.json()

[{'COD': 'ECP320',
  'Nombre': 'Total Nacional. Todas las edades. Total. Población. Número. ',
  'T3_Unidad': 'Personas',
  'T3_Escala': ' ',
  'Notas': [{'texto': 'https://www.ine.es/dyngs/INEbase/es/operacion.htm?c=Estadistica_C&cid=1254736177095&menu=ultiDatos&idp=1254735572981#~# https://www.ine.es/consul/serie.do?s=ECP320&nult=100&TB_iframe=true&position=center&width=700&height=500&L=0&nocab=1',
    'Fk_TipoNota': 6,
    'textoTipo': None}],
  'Data': [{'Fecha': '2025-01-01T00:00:00.000+01:00',
    'T3_TipoDato': 'Definitivo',
    'T3_Periodo': '1 de enero de',
    'Anyo': 2025,
    'Valor': 49128297.0}]}]